# Open Gym Analytics — Análisis exploratorio de datos

## Objetivo
Analizar patrones de uso de espacios deportivos comunitarios para identificar actividades, horarios y centros con mayor utilización.

## Importante sobre las métricas
El dashboard original de Power BI utiliza en varias visualizaciones el **recuento de registros** de `Total Asistentes`. En este notebook se distinguen dos métricas:

- **Registros:** cantidad de filas del dataset.
- **Asistentes:** suma de la columna `total_asistentes`.

Esta distinción evita confundir frecuencia de registros con cantidad de personas registradas.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
df = pd.read_csv("../data/processed/open-gym-clean.csv")
df.head()


## 1. Exploración inicial

In [ ]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
df.dtypes


In [ ]:
missing = df.isna().sum().sort_values(ascending=False).to_frame("valores_nulos")
missing["porcentaje"] = (missing["valores_nulos"] / len(df) * 100).round(2)
missing


## 2. Variables derivadas
Se utilizaron `fecha`, `hora`, `dia_semana` y `duracion_horas` para facilitar el análisis temporal.


In [ ]:
df[["fecha_hora_apertura","fecha_hora_cierre","fecha","hora","dia_semana","duracion_horas"]].head()


## 3. KPIs principales

In [ ]:
kpis = pd.Series({
    "Total de asistentes registrados": int(df["total_asistentes"].sum()),
    "Número de registros": int(len(df)),
    "Promedio de asistentes por registro": round(df["total_asistentes"].mean(), 2),
    "Actividad con más registros": activity_records.index[0],
    "Actividad con más asistentes": activity_attendance.index[0],
    "Hora con más registros": int(hour_records.index[0]),
    "Hora con más asistentes": int(hour_attendance.index[0]),
    "Centro con más asistentes": center_attendance.index[0],
})
kpis


## 4. Actividades: registros vs. asistentes
Esta comparación es importante porque el dashboard de Power BI presenta el recuento de registros, mientras que aquí también se analiza la suma de asistentes.


In [ ]:
activity = pd.DataFrame({
    "registros": activity_records,
    "asistentes": activity_attendance
}).sort_values("asistentes", ascending=False)

activity


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
activity["asistentes"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Asistentes totales por actividad")
ax.set_xlabel("Total de asistentes")
ax.set_ylabel("Actividad")
plt.tight_layout()
plt.show()


## 5. Horarios
Se analizan tanto el número de registros como la suma de asistentes por hora.


In [ ]:
hour = pd.DataFrame({
    "registros": hour_records,
    "asistentes": hour_attendance
}).sort_index()

hour


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
hour["asistentes"].plot(kind="bar", ax=ax)
ax.set_title("Asistentes totales por hora")
ax.set_xlabel("Hora")
ax.set_ylabel("Total de asistentes")
plt.tight_layout()
plt.show()


## 6. Centros comunitarios

In [ ]:
center = pd.DataFrame({
    "registros": df.groupby("centro_comunitario")["total_asistentes"].count(),
    "asistentes": center_attendance
}).sort_values("asistentes", ascending=False)

center


## 7. Distribución registrada por género

In [ ]:
gender = pd.Series({
    "Hombres": df["total_hombres"].sum(),
    "Mujeres": df["total_mujeres"].sum()
})
gender


In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
ax.pie(gender.values, labels=gender.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Distribución registrada por género")
plt.tight_layout()
plt.show()


## 8. Hallazgos
- El dataset contiene **28.038 registros** y acumula **171.413 asistentes** en la columna `total_asistentes`.
- **Basketball** es la actividad con más registros (8.971), seguida por **Open Gym** (7.643).
- Si la métrica es la suma de `total_asistentes`, **Pickleball** y **Badminton** acumulan los mayores valores.
- Por suma de asistentes, las horas con mayor volumen son **22:00, 13:00 y 14:00**.
- **BPCC** es el centro con mayor suma de asistentes.
- El dashboard de Power BI debe interpretarse teniendo presente que varias visualizaciones muestran **recuento de registros**, no suma de asistentes.


## 9. Conclusión
El proyecto permite analizar la utilización de espacios deportivos desde dos perspectivas complementarias: frecuencia de registros y volumen acumulado de asistentes. La separación de estas métricas mejora la precisión de las conclusiones y evita interpretar un recuento de filas como si fuera una cantidad de personas.
